<h1 style="text-align: center;">TÉCNICO EM CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Roteiro do Desenvolvendo Juntos</h1>
<br>
<br>

**Componente:** Fundamentos de Ambientes e Arquitetura de Dados
<br>
**Unidade Curricular:** Introdução ao PySpark e Processamento Distribuído
<br>
**Tema da Semana:** Leitura de dados
<br>
**Semana 17 – Aula 1**
<br>
**Aula 1:** Leitura de CSV e JSON


## Objetivos
- Compreender como o PySpark lê arquivos **CSV** e **JSON**.
- Utilizar `spark.read.csv()` e `spark.read.json()` para gerar DataFrames.
- Configurar a leitura com `.option()` e analisar o impacto do `inferSchema`.
- Comparar o **schema** de um DataFrame com e sem inferência de tipos.


## Lista de Materiais

- Computador com Anaconda (Jupyter Notebook) ou VS Code.
- Ambiente com PySpark configurado.
- Arquivo **DADOSANO2C6B3S17A1_desenvolvendo_juntos.ipynb.**
- Arquivo auxiliar: **DADOSANO2C6B3S17A1_auxiliar.csv** (na mesma pasta deste notebook).

### Etapa 1: Iniciando a SparkSession
A **SparkSession** é o ponto de entrada do PySpark.  
Nesta aula, vamos iniciar a sessão para poder usar `spark.read` e carregar arquivos como DataFrames.


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("S17A1_Leitura_CSV_JSON")
    .master("local[*]")  # usa todos os núcleos disponíveis no computador
    .getOrCreate()
)

print("SparkSession iniciada:", spark.version)

SparkSession iniciada: 4.1.1


### O que fizemos aqui?
- Criamos a SparkSession com um nome de aplicação.
- Definimos o modo **local** para executar no próprio computador.
- A variável `spark` será usada para **ler arquivos** e criar DataFrames.


### Etapa 2: Leitura simples de um CSV
Vamos ler o arquivo **DADOSANO2C6B3S17A1_auxiliar.csv** usando `spark.read.csv()`.

Primeiro, faremos uma leitura **simples**, sem configurações adicionais, para observar o comportamento padrão do Spark.

In [2]:
df_csv_simples = spark.read.csv("DADOSANO2C6B3S17A1_auxiliar.csv")

print("Leitura simples concluída.")
df_csv_simples.show(5, truncate=False)

Leitura simples concluída.
+---+-------+-----+-------+
|_c0|_c1    |_c2  |_c3    |
+---+-------+-----+-------+
|id |nome   |idade|salario|
|1  |Ana    |23   |2500.5 |
|2  |Bruno  |35   |3800.0 |
|3  |Carlos |29   |3100.75|
|4  |Daniela|41   |5200.0 |
+---+-------+-----+-------+
only showing top 5 rows


### O que observar?
- Sem opções, o Spark:
  - **não** interpreta a primeira linha como cabeçalho;
  - lê tudo como **string**;
  - cria nomes de colunas genéricos (ex.: `_c0`, `_c1`, ...).

### Etapa 3: Leitura de CSV com opções
Agora, vamos configurar a leitura com `.option()`:

- `header = true` → usa a primeira linha como nome das colunas  
- `inferSchema = true` → tenta identificar o tipo correto de cada coluna  
- `sep` → define o separador (ajuste se seu CSV não for vírgula)

> Se o seu arquivo usar `;` como separador, altere `sep` para `;`.


In [3]:
df_csv = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")  # troque para ";" se necess?rio
    .csv("DADOSANO2C6B3S17A1_auxiliar.csv")
)

print("Leitura com op??es conclu?da.")
df_csv.show(5, truncate=False)


Leitura com op??es conclu?da.
+---+-------+-----+-------+
|id |nome   |idade|salario|
+---+-------+-----+-------+
|1  |Ana    |23   |2500.5 |
|2  |Bruno  |35   |3800.0 |
|3  |Carlos |29   |3100.75|
|4  |Daniela|41   |5200.0 |
|5  |Eduardo|30   |2900.25|
+---+-------+-----+-------+



### O que fizemos nesta etapa?

- Utilizamos .option("header", "true") para indicar que a primeira linha contém os nomes das colunas.

- Utilizamos .option("inferSchema", "true") para que o Spark tente identificar os tipos de dados.

- Utilizamos .option("sep", ",") para definir o separador do arquivo.

- Em seguida, lemos o CSV e visualizamos os dados.

### Etapa 4: Comparando o schema (com e sem inferSchema)
Nesta etapa, vamos comparar os tipos das colunas.

**Pergunta norteadora:**  
> O que muda no tipo das colunas quando ativamos `inferSchema`?

In [4]:
print("Schema - leitura simples (sem inferSchema):")
df_csv_simples.printSchema()

print("\nSchema - leitura com inferSchema:")
df_csv.printSchema()

Schema - leitura simples (sem inferSchema):
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)


Schema - leitura com inferSchema:
root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- salario: double (nullable = true)



### Interpretação rápida
- Sem `inferSchema`, colunas aparecem como **string**.
- Com `inferSchema`, colunas numéricas tendem a virar **integer** ou **double**.
- Isso evita erros quando você fizer **cálculos**, **filtros** e **agregações**.

### Etapa 5: Leitura de JSON
JSON é muito comum em APIs e sistemas web.  
Para garantir que todos consigam executar, vamos criar um arquivo JSON simples (se ele não existir) e então ler com `spark.read.json()`.


In [5]:
import os, json

json_path = "DADOSANO2C6B3S17A1_auxiliar.json"

df_json = spark.read.option("multiLine", "true").json(json_path)

print("Leitura de JSON conclu?da.")
df_json.show(truncate=False)


Leitura de JSON conclu?da.
+---+-----+-------+-------+
|id |idade|nome   |salario|
+---+-----+-------+-------+
|1  |23   |Ana    |2500.5 |
|2  |35   |Bruno  |3800.0 |
|3  |29   |Carlos |3100.75|
|4  |41   |Daniela|5200.0 |
|5  |30   |Eduardo|2900.25|
+---+-----+-------+-------+



### Etapa 6: Conferindo o schema do JSON
O JSON pode conter estruturas mais complexas (campos aninhados).  
Aqui, vamos apenas verificar o schema gerado automaticamente.


In [6]:
df_json.printSchema()

root
 |-- id: long (nullable = true)
 |-- idade: long (nullable = true)
 |-- nome: string (nullable = true)
 |-- salario: double (nullable = true)

